In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7796511384929194, 'n_it': 0.37510785080683545}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.284972947742334, 14.097865897601356, 14.601586341883099, 15.817790785005162, 16.488196192243812, 14.108712913704315, 14.608735990809151, 13.687215687366061, 15.235878980125449, 13.930343459190201, 14.143941506259022, 14.009921487876204, 15.309112313939606, 13.72420420087262, 13.765106323217493, 17.046101922978934, 16.975888797444153, 13.691567785202633, 14.845430370361662, 13.664793269030278, 17.021964267472303, 13.785245494566034, 14.463547859375259, 16.628172655514636, 16.448512126090847, 13.854028371854575, 13.618514174536008, 14.768068640832992, 14.012840891783048, 14.315495833291033, 17.062500407494028, 16.58569954924469, 13.716759054574206, 14.017692568900868, 14.65409089510924, 13.784318170867305, 14.337075868237953, 13.644943991739583, 13.851801880891701, 14.92672015336209, 16.16420720622449, 14.216894434444203, 17.811197077454043, 15.175368953455395, 13.74562406862322, 13.623492830962402, 14.621824529958346, 17.341914111308984, 13.821786438893366, 13.856125328462348, 16.87

In [5]:
np.average(y_max_arr)

np.float64(14.80492721703132)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M05/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)